<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# What machine learning is

**[What machine learning is](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/)** · Machine Learning: From Problem to Reliable Model · Module 1, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** tell a rule that a person writes from a rule that a program learns from examples, and to use the words model, training and prediction.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Python functions, `if` and lists (the [Python course](https://learning.nextia-ai.com/courses/python/)), and how to read a CSV file into a pandas DataFrame (the [SQL course](https://learning.nextia-ai.com/courses/sql/)). |
| **You do not need** | A local project, or any knowledge of machine learning. This is the first code of the course. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M01-L01-what-machine-learning-is/what-machine-learning-is-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Twelve messages with the right answer

A program must decide if a message is spam. Run the next cell. It makes a small table of 12 made-up messages. The column `spam` has the right answer for each one: 1 for spam, 0 for a normal message. A person read each message and wrote the answer.

The [lesson page](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/#a-tiny-example-twelve-messages) explains each step in more detail.

In [ ]:
messages = pd.DataFrame({
    "text": [
        "WIN a free phone now, click here",
        "You are a winner! Claim your free prize",
        "Cheap watches, best price, buy now",
        "Free gift card for you, click the link",
        "Earn money fast from home, click now",
        "Claim your prize today, offer ends soon",
        "Can we meet for lunch tomorrow?",
        "Your order 4417 has shipped",
        "Here are the notes from today's meeting",
        "Is the free car park open on Sunday?",
        "Happy birthday! See you on Saturday",
        "The invoice for March is attached",
    ],
    "spam": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})
print(len(messages), "messages,", messages["spam"].sum(), "of them spam")

> **Check.** You should see `12 messages, 6 of them spam`.

## 2. Way 1: write the rule yourself

The first way is the traditional one: a person writes the rule. Many spam messages have the word "free" or "winner", so the rule says: spam if the message has one of these words.

> **Predict.** How many of the 12 messages will the rule get right? Write a number, then run the cell.

In [ ]:
def rule(text):
    """Spam (1) if the message has the word "free" or "winner"."""
    text = text.lower()
    return int("free" in text or "winner" in text)


messages["rule"] = messages["text"].map(rule)
print(messages)
print("Right:", (messages["rule"] == messages["spam"]).sum(), "of", len(messages))

> **Check.** You should see `Right: 8 of 12`. The rule misses three spam messages (rows 2, 4 and 5), and it marks the message about the free car park (row 9) as spam.

To fix the rule, you would add more words: "cheap", "prize", "money", and so on. Each new word can also catch normal messages, as "free" caught the message about the car park. And spammers change their words every week. A rule list for real email grows to thousands of rules, and nobody can keep it right.

## 3. Way 2: let the program learn the rule from the examples

The second way is **machine learning**. You do not write the rule. You give the program the examples **and** their right answers, and it finds a rule itself.

Run the next cell. It defines `new_spam_model()`, a helper that makes an empty model. The model counts the words of each message. During learning, it finds how much more often each word appears in spam than in normal messages. You do not need its details now.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline


def new_spam_model():
    """An empty model: it counts the words, then learns which words point to spam."""
    return make_pipeline(CountVectorizer(), MultinomialNB())

Now the two steps that every machine learning model has:

- **Training**: `model.fit(examples, answers)` learns the rule from the examples.
- **Prediction**: `model.predict(new_examples)` uses the learned rule on messages.

> **Predict.** The model learns from these 12 messages. How many of the same 12 will it get right? Then run the cell.

In [ ]:
model = new_spam_model()
model.fit(messages["text"], messages["spam"])  # training: learn from the examples

messages["model"] = model.predict(messages["text"])  # prediction
print("Right:", (messages["model"] == messages["spam"]).sum(), "of", len(messages))

> **Check.** You should see `Right: 12 of 12`.

This is not yet a fair test: the model saw these 12 answers while it learned. A student who has seen the answers also gets every question right. The real test is messages that the model has never seen.

## 4. Six new messages

Run the next cell. It makes six new messages that neither the rule nor the model has seen. Then it asks both for an answer.

> **Predict.** Which way will do better on the new messages: the written rule or the model?

In [ ]:
new = pd.DataFrame({
    "text": [
        "Click now to claim your cash prize",
        "Buy cheap phones, best offer today",
        "Lunch on Friday?",
        "Your parcel has shipped",
        "Fr3e g1ft for you, click here",
        "The notes from the meeting are attached",
    ],
    "spam": [1, 1, 0, 0, 1, 0],
})
new["rule"] = new["text"].map(rule)
new["model"] = model.predict(new["text"])
print(new)
print("Rule right:", (new["rule"] == new["spam"]).sum(), "of 6;",
      "model right:", (new["model"] == new["spam"]).sum(), "of 6")

> **Check.** You should see `Rule right: 3 of 6; model right: 6 of 6`.

The rule misses all three spam messages, because none of them has "free" or "winner". The spammer wrote "Fr3e" to get past it. The model uses many clues together ("click", "claim", "prize", "cheap"), so one changed word does not stop it.

Do not trust six messages too much: with so few, part of this is luck. A real spam filter learns from millions of messages, and it still makes some mistakes. The important difference is this: **to improve the model, you give it more examples. To improve the rule, a person must write more rules.**

## 5. A first look at Larkfield's history

Larkfield's help desk has the same kind of history: thousands of past tickets, each with its right answer. The column `escalated_72h` is 1 if a senior agent took over the ticket within 72 hours. Run the next cell. It reads the train tickets and shows how often tickets of each priority escalated.

In [ ]:
train = pd.read_csv("data/train.csv")
print(len(train), "past tickets,", train["escalated_72h"].sum(), "escalated")
print(train.groupby("priority")["escalated_72h"].mean().round(3))

> **Check.** You should see `16939 past tickets, 1983 escalated`, then `0.291` for priority 1, `0.120` for priority 2 and `0.067` for priority 3.

Priority tells you something: 29.1% of priority 1 tickets escalated, and only 6.7% of priority 3 tickets. But no single column decides. Most escalations (710 + 580 of 1,983) come from priority 2 and 3 tickets. To use many clues together, Larkfield needs a rule learned from the examples. That is the project of this course.

## 6. Your turn: write a rule, or learn from examples?

> **Your turn.** For each task, write `"rule"` if a person can write the exact rule, or `"examples"` if a program should learn it from examples. Then run the check cell.

1. Give free delivery to every order above $50.
2. Read handwritten postcodes on parcels.
3. Suggest films to a viewer from what similar viewers watched.
4. Add the sales tax to an invoice.
5. Estimate when a taxi arrives, in city traffic.
6. Check that a new password has at least 12 characters.

In [ ]:
ways = {
    "q1": "",
    "q2": "",
    "q3": "",
    "q4": "",
    "q5": "",
    "q6": "",
}

In [ ]:
if any(value not in ("rule", "examples") for value in ways.values()):
    print('Not yet: write "rule" or "examples" for each task.')
else:
    expect_hash("the dictionary ways", ways, '27706eb1f89b5bf1')

Ask two questions for each task. Can a person say the rule exactly, in a few lines? Then write it. Is the pattern too complex to write, and do many examples with answers exist? Then learn it.

## 7. Change one thing: a new kind of spam

Spammers change their messages all the time. Run the next cell. It asks the model about a new kind of spam: a "holiday voucher".

In [ ]:
tricky = "Your holiday voucher is waiting, reply today"
print("The model says:", model.predict([tricky])[0])

> **Check.** You should see `The model says: 0`. The model has never seen a message like this, so it says "normal".

With a written rule, you would now write a new rule. With machine learning, you add examples.

> **Your turn.** Add two new spam messages about vouchers to `more_examples`, each with the answer 1. Write them as you think a spammer would. Run the cell, then the check cell. The check passes when the retrained model calls `tricky` spam (1) and still calls a normal message normal (0).

In [ ]:
more_examples = pd.DataFrame({
    "text": [
        "...",  # a spam message about a voucher
        "...",  # another one
    ],
    "spam": [1, 1],
})
bigger = pd.concat([messages[["text", "spam"]], more_examples], ignore_index=True)
model2 = new_spam_model()
model2.fit(bigger["text"], bigger["spam"])
print("The new model says:", model2.predict([tricky])[0])

In [ ]:
normal = "Congratulations on your new job! See you at the party"
if model2.predict([tricky])[0] == 1 and model2.predict([normal])[0] == 0:
    print("Check passed: the model learned the new kind of spam, and a normal message stays normal.")
elif model2.predict([tricky])[0] != 1:
    print("Not yet: the model still calls the voucher message normal. Use more of its words in your examples.")
else:
    print("Not yet: the model now calls a normal message spam. Make your examples more like spam.")

No person wrote a new rule. You gave two examples with the right answer, trained again, and the model changed. This is the main idea of machine learning: **the examples teach the rule**.

## 8. Failure case: a model for an answer that a rule already knows

Tomás wants to use machine learning everywhere. He trains a model to say if an order is above $500. It learns from nine example orders, each with the right answer. The model copies the answer of the most similar example order.

> **Predict.** The rule `order_value > 500` is exact. Will the model agree with it on four new orders: 490, 505, 520 and 640?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

orders = pd.DataFrame({"order_value": [120, 260, 380, 455, 470, 560, 610, 750, 900]})
orders["above_500"] = (orders["order_value"] > 500).astype(int)
copycat = KNeighborsClassifier(n_neighbors=1)  # copies the answer of the most similar order
copycat.fit(orders[["order_value"]], orders["above_500"])

check = pd.DataFrame({"order_value": [490, 505, 520, 640]})
check["rule"] = (check["order_value"] > 500).astype(int)
check["model"] = copycat.predict(check[["order_value"]])
print(check)

> **Check.** You should see the model say `0` for the order of 505, where the rule says `1`. The other three agree.

The most similar example of 505 is 470, so the model copies "not above". The rule is never wrong, costs nothing to train, and anyone can read it. When a person can write the exact rule, write the rule. Use machine learning only when the rule is too complex to write and examples with answers exist.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this notebook, a written rule got 8 of 12 messages right and missed all the new spam. A model trained on the same 12 examples got the new messages right, and it learned a new kind of spam from two more examples. But for "order above $500", the written rule is exact and the model is not. The words: **training** (`fit`) learns a **model** from examples with answers, and **prediction** (`predict`) uses the model on new cases.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m01/what-machine-learning-is/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.